# Parameter setzen:

In [ ]:
import os
import numpy as np
import sys
from scipy.io import loadmat
from pathlib import Path

# GPU wählen
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

# Run-Config
model = "Sim_Lesion_double_xy-f" #fn_251009_WB_DMI_P01_pos4 Sim_Lesion_double_xyf
batch_size = 600

model_input = "../datasets/Simulated_Lesion_double_1_normalized_tMPPCA_5D/data.npy"  
#"../datasets/Simulated_Lesion_double_1_normalized_tMPPCA_5D/data.npy"  
#"../datasets/fn_251009_WB_DMI_P01_4pos/output_4/CombinedCSI.mat"
GT_path = "../datasets/Simulated_Lesion_GT_double_normalized/data.npy"

# ---- Comparisons definieren ----
comparison_paths = [
    model_input,
    GT_path,
]

Titles = [
    "Deep",
    "No Denoising",
    "GT",
]

# ---- Daten laden (einmalig!) ----
Data = []

for p in comparison_paths:
    p = Path(p)

    if p.suffix == ".npy":
        arr = np.load(p)

    elif p.suffix == ".mat":
        mat = loadmat(p)
        arr = np.asarray(mat["csi"]["Data"][0,0])

    else:
        raise ValueError(f"Unsupported file format: {p}")

    Data.append(arr)

# Inference

In [ ]:
import matplotlib.pyplot as plt

sys.path.append(os.path.abspath(".."))
sys.path.append(os.path.abspath("../src"))

from denoising.config.load import load_yaml
from denoising.config.build import build_config
from denoising.inference.api import infer
from src.denoising.figures.EvalBeforeFitting import *

config_path = f"../trained_models/{model}/train.yaml"

cfg = build_config(load_yaml(config_path))

denoised, meta = infer(
    cfg=cfg,
    ckpt_path=f"../trained_models/{model}/checkpoints/last.pt",
    input_path=model_input,   # exakter Pfad zur Datei
    batch_size=batch_size
)

Data.insert(0, denoised)

# auch fft
Data_ft = [np.fft.fftshift(np.fft.fft(d, axis=3), axes=3) for d in Data]

# Optional als matlab datei speichern

In [ ]:
# import numpy as np 

# out_data = np.load("data_denoised.npy")

# from scipy.io import loadmat, savemat

# savemat('sf_brain_DMI_HC_pilot_Combined_Model.mat', {'Data': out_data})

# # Für SIMS:

# ### CAREFUL DISTINGUISH BETWEEN LESION AND NO LESION

# #/Lade Par und mask aus der B0_1.mat-Datei
# b0_data = loadmat('../datasets/Simulated_Lesion_GT/Lesion_120pts.mat')  #Simulated_Lesion_GT/Lesion_120pts.mat  Simulated_ground_truth/B0_1.mat
# par = b0_data['csi_data_lr']['Par'][0, 0]
# mask = b0_data['csi_data_lr']['mask'][0, 0]

# # Deine Datensätze
# data_dicts = {
#     f'Lesion_double_deep_tmppca_6_retrained.mat': denoised
#     #f'Lesion_double_deep_noisy_6_uncorrelated.mat': noisy_data
#     #'Lesion_gt.mat': tgt_data
# }

# # Für jede Datei speichern
# for filename, data in data_dicts.items():
#     savemat(filename, {
#         'csi_data_lr': {
#             'Data': data,
#             'Par': par,
#             'mask': mask
#         }
#     })

# Compare spectral slice

In [ ]:
fig, axes = plot_z_slices(
    Data_ft,
    Titles,
    t=50,
    T=7,
    share_clim="per_column",
)

plt.show()

# Compare Spectra

In [ ]:
_ = plot_voxel_spectra_over_z(Data_ft, Titles, x=8, y=10, T=7, z_max=21)

plt.show()

# Compare spectral peaks

# Compare average spectra
Here I compare the average spectrum over time (which is a high SNR estimate)

In [ ]:
Avg = [np.mean(d, axis=(0,1,2)) for d in Data_ft]

_ = plot_average_spectra_over_T(
    Avg,
    Titles,
    n_cols=2,
)
plt.show()